In [1]:
!pip install arch
!pip install pyvinecopulib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import pyvinecopulib as pv
from scipy.stats import genpareto
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [3]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

In [4]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [20]:
class MarginalTailModel:
  def __init__(self, sim_params, ticker):
    self.sim_params = sim_params
    self.ticker = ticker
    self.params = None

  def fit_dist(self, r):
    r = r.dropna().to_numpy()
    q_upper = 1 - self.sim_params.q_upper
    q_lower = self.sim_params.q_upper

    u_upper = np.percentile(r, q_upper * 100)
    u_lower = np.percentile(r, q_lower * 100)

    upper_tail = r[r > u_upper]
    lower_tail = r[r < u_lower]

    c_U, _, scale_U = genpareto.fit(upper_tail - u_upper, floc=0)
    c_L, _, scale_L = genpareto.fit(u_lower - lower_tail, floc=0)

    p_l = np.mean(r <= u_lower)
    p_u = np.mean(r >= u_upper)

    lower_mask = r < u_lower
    upper_mask = r > u_upper
    body_mask = ~lower_mask & ~upper_mask

    u_resid = np.zeros_like(r, dtype=float)

    if np.any(lower_mask):
      cdf = 1 - genpareto.cdf(u_lower - r[lower_mask], c_L, scale=scale_L)
      u_resid[lower_mask] = p_l * cdf

    if np.any(body_mask):
      body = r[body_mask]
      ranks = pd.Series(body).rank(method='average').to_numpy()
      cdf = p_l + (1 - p_l - p_u) * (ranks / (len(body) + 1))
      u_resid[body_mask] = cdf

    if np.any(upper_mask):
      cdf = (1 - p_u) + p_u * genpareto.cdf(r[upper_mask] - u_upper, c_U, scale=scale_U)
      u_resid[upper_mask] = p_u * cdf

    self.params = {
        "c_L": c_L,
        "scale_L": scale_L,
        "c_U": c_U,
        "scale_U": scale_U,
        "p_l": p_l,
        "p_u": p_u,
        "u_lower": u_lower,
        "u_upper": u_upper,
        "ecdf_data": body
    }

    return u_resid

  def inverse_cdf(self, u_resid):
    z_resid_final = u_resid

    for i in range(u_resid.shape[0]):
      u_resid_i = u_resid[:, i]
      real_col = np.zeros_like(u_resid_i)

      lower_mask = u_resid_i < self.params.p_l
      upper_mask = u_resid_i > self.params.pu
      body_mask = (~lower_mask) & (~upper_mask)
      p_u = self.params.p_u
      p_l = self.params.p_l

      if np.any(upper_mask):
        ppf_l = (
            self.params.upper +
            genpareto.ppf(
                (u_resid_i[upper_mask] - (1 - p_u))/p_u,
                self.params.c_U,
                scale=self.params.scale_U
            )
        )
        real_col[upper_mask] = ppf_l

      if np.any(lower_mask):
        ppf_u = (
            self.params.lower -
            genpareto.ppf(
                1 - u_resid_i[lower_mask]/p_l,
                self.params.c_U,
                scale=self.params.scale_U
            )
        )

        real_col[lower_mask] = ppf_u

      if np.any(body_mask):
        body = (u_resid_i[body_mask] - p_l) / (1 - p_l - p_u) * 100
        body_percentiles = np.clip(body_percentiles, 0, 100)
        ppf_b = np.percentile(self.params.ecdf_data, body_percentiles)
        real_col[body_mask] = ppf_b

      z_resid_final[:, i] = real_col

    return z_resid_final










In [19]:
class GARCHEVTCOPULA:
  def __init__(self, sim_params, debug:bool=False, **kwargs):
    self.sim_params = sim_params
    self.debug = debug
    self.copula = None
    self.best_models = {}
    self.best_fits = {}
    self.best_params = {}
    self.scale_factor = 100
    self.marginal_distributions = {}

  def _fit_model(self, r):
    r_scaled = r * self.scale_factor
    max_bic = -np.inf
    for lags in range(self.sim_params.lags):
        model = arch_model(
          r_scaled,
          mean="AR",
          lags=lags,
          vol="GARCH",
          p=1,
          q=1,
          dist='studentst'
        )

        fit = model.fit(disp='off', show_warning=False)

        if fit.bic > max_bic:
          max_bic = fit.bic
          best_model = model
          best_fit = fit
          best_lags = lags

    return best_model, best_fit, best_lags


  def _fit_arma_garch(self, returns):
    print("------------- Fitting AR-GARCH -------------")
    log_returns = np.log1p(returns)
    filtered_resid = pd.DataFrame(index=returns.index, columns=returns.columns)

    for i, col in enumerate(log_returns.columns):
      best_model, best_fit, best_lags = self._fit_model(log_returns.col)

      self.best_models[col] = best_model
      self.best_fits[col] = best_fit
      self.best_params[col] = best_fit.params

      filtered_resid[col] = best_fit.std_resid

    return filtered_resid


  def _get_pseudo_observations(self, residuals):
    print("------------ Uniform Residuals -------------")
    u_resid = pd.DataFrame(index=residuals.index, columns=residuals.columns)
    for ticker in residuals.columns:
      model = MarginalTailModel(self.sim_params, ticker)
      u_resid_i = model.fit_dist(residuals[ticker])
      self.marginal_distributions[ticker] = model

      u_resid[ticker] = u_resid_i

    return u_resid




  def _inverse_semi_parametric_cdf(self, uniform_samples):
    print("------------ Inverse Semi Parametric CDF -------------")


  def _fit_vine_copula(self, residuals):
    print("-------------- Fitting Copula --------------")

  def fit(self, returns, save=True):
    z_resid = self._fit_ar_garch(returns)
    self.cols = z_resid.columns
    u_resid = self.get_uniform_residuals(z_resid)

    self._fit_vine_copula(u_resid)

  def generate_sample(self):
    pass



In [6]:
class CVaREngine(GARCHEVTCOPULA):
  def __init__(self, cvar_engine_params, debug:bool=False, **kwargs):
    self.debug = debug
    self.cvar_params = cvar_engine_params

  def fit_garch_evt_copula(self, returns):
    pass
